# License

This notebook is part of the `fslbfi` project.

Copyright (c) 2026 fslbfi

This notebook is licensed under the MIT License. See the file
`LICENSE-NOTEBOOKS` in this repository for the full text.

# Notebook 4 — ProtoNet Episodic Training (Experiments 2 & 3)
## BFI-Based Few-Shot Binary Occupancy Detection

---

**Goal:** Train a ProtoNet-style few-shot learner on BFI window tensors and evaluate
cross-device and cross-position generalisation with K ∈ {1, 3, 5, 10} labeled samples
per class.

**What this notebook does:**
1. Loads all processed `.npy` tensors from Notebook 1 as per-trace records (W varies per trace).
2. Optionally warm-starts the encoder from the Notebook 3 checkpoint.
3. Implements `sample_episode_stratified` — stratified episode sampler that draws equal
   windows per device per class, preventing any single device from dominating episodes.
4. Implements ProtoNet loss (prototype computation → squared-Euclidean distance → softmax CE).
5. **Experiment 2:** Cross-Device FSL — leave-one-device-out, 3 target devices.
6. **Experiment 3:** Cross-Position FSL — leave-one-session-out per device, 3 folds × 3 devices.
7. Reports per-K accuracy, precision, recall, F1, and AUC-PR.
8. Saves final encoder checkpoint, JSON results files, and meta-train loss logs.
9. Generates publication-ready figures from saved results.

**Prerequisites:** Notebooks 1–3 must have been run successfully.

**ProtoNet core idea** (per episode):
```
support set → encoder → class prototypes (mean embedding per class)
query set   → encoder → query embeddings
             → neg. squared-Euclidean distances → softmax CE loss → backprop
```
Only the encoder is updated during meta-training; the prototype head has no learnable parameters.

**Stratified sampling rationale:**
Devices have different BFI rates — X7 captures more frames per second than M7 or X300.
Without stratification, pooling all training windows and drawing uniformly would over-represent
higher-rate devices within each episode, biasing prototypes toward their feature distribution.
The stratified sampler enforces equal per-device, per-class contributions inside every episode,
making meta-training device-agnostic. All windows remain available; none are permanently discarded.

**Evaluation cap (optional robustness check):**
By default `Q_EVAL_CAP = None` and fair comparison is achieved via macro-averaging per device. 
If set to an integer, the query pool per device is capped at that value per fold;
use this to verify that results are stable regardless of pool size.

---
## 0. Configuration

All hyperparameters from `bfi-fsl-overview.pdf` §6.2 are set here and **not**
repeated elsewhere in the notebook.

In [12]:
import re, json, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import defaultdict
from sklearn.metrics import average_precision_score

# ── Paths ──────────────────────────────────────────────────────────────────────
PROCESSED_DIR  = Path('data/processed')  # Notebook 1 output
CHECKPOINT_DIR = Path('checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Derive constants from one saved file ─────────────────────────────────────
_probe = next(PROCESSED_DIR.rglob('p*.npy'), None)
if _probe is None:
    raise FileNotFoundError(f'No p*.npy files found under {PROCESSED_DIR}. Run Notebook 1 first.')
_arr = np.load(_probe)  # (W, T, K, C)
_, WINDOW_SIZE, TARGET_SUBCARRIERS, N_CHANNELS = _arr.shape
TARGET_NSS = 1  # keep first spatial stream only (M7: Nss=1; X7/X300: Nss=2 -> stream 0)
LABEL_MAP = {'empty': 0, 'stationary': 1, 'moving': 1}

# ── Architecture — mirror Notebooks 2 & 3 ─────────────────────────────────────
EMBED_DIM = 64

# ── Meta-training hyperparameters (§6.2) ──────────────────────────────────────
N_WAY            = 2    # binary: empty vs. occupied
K_SHOT_TRAIN     = 5    # support samples per class during meta-training
Q_QUERY          = 15   # query samples per class per device during meta-training
EPISODES_PER_EPOCH = 200  # episodes per epoch (200 for production, 100 for test)
META_EPOCHS      = 100  # max meta-training epochs (100 for production, 50 for test)
META_LR          = 1e-3
META_PATIENCE    = META_EPOCHS  # effectively disables early stopping — runs full fixed-epoch schedule

# ── K-shot values evaluated at inference ──────────────────────────────────────
K_SHOT_EVAL = [1, 3, 5, 10]

# ── Evaluation query cap ───────────────────────────────────────────────────────
# Set dynamically per fold to min session window count across devices in that fold.
# Ensures each device contributes the same number of query windows → fair metric aggregation.
# Override with an integer to hard-cap across all folds (None = dynamic, recommended).
Q_EVAL_CAP = None  # None → computed per fold

# ── Optional: warm-start encoder from Notebook 3 ──────────────────────────────
# Set to None to train from random initialisation (most defensible for thesis).
# A warm-started encoder would contain supervised information about the target
# domain, compromising the cross-device / cross-position generalisation claim.
WARMSTART_CKPT = None

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_SEED = 67
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

TORCH_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Global training log accumulator ───────────────────────────────────────────
# Each call to meta_train() appends its epoch-loss-history dict under its label.
# Saved to results/meta_train_logs.json at the end of the notebook.
META_TRAIN_LOGS = {}

print('Configuration loaded.')
print(f'  Device       : {TORCH_DEVICE}')
print(f'  Input shape  : (N, {N_CHANNELS}, {WINDOW_SIZE}, {TARGET_SUBCARRIERS})')
print(f'  Episode cfg  : {N_WAY}-way {K_SHOT_TRAIN}-shot Q={Q_QUERY} per device per class')
print(f'  Meta-train   : {META_EPOCHS} epochs × {EPISODES_PER_EPOCH} episodes')
print(f'  K-shot eval  : {K_SHOT_EVAL}')
print(f'  Warm-start   : {WARMSTART_CKPT}')
print(f'  Q_EVAL_CAP   : {Q_EVAL_CAP} (None = dynamic per fold)')

Configuration loaded.
  Device       : cuda
  Input shape  : (N, 5, 5, 234)
  Episode cfg  : 2-way 5-shot Q=15 per device per class
  Meta-train   : 100 epochs × 200 episodes
  K-shot eval  : [1, 3, 5, 10]
  Warm-start   : None
  Q_EVAL_CAP   : None (None = dynamic per fold)


---
## 1. Load All Processed Data (Per-Trace Records)

Notebook 1 saves one `.npy` per trace; W varies across files because the global frame
cap was removed. We load each file as a separate record and keep them distinct.
The stratified sampler indexes directly into per-device, per-class lists of windows,
so we must **not** flatten everything into one concatenated array before sampling.

In [ ]:
def load_all_records(processed_dir: Path) -> list:
    """
    Load every p*.npy from processed_dir.

    Returns
    -------
    records : list of dict
        Each dict has:
          'windows'  : np.ndarray (W, T, K, C) — W varies per trace
          'label'    : int  (0 = empty, 1 = occupied)
          'device'   : str  e.g. 'M7'
          'session'  : int  session number (1, 2, or 3)
          'scenario' : str  e.g. 'empty'
          'file'     : str  filename
    """
    records = []
    for fpath in sorted(processed_dir.glob('**/p*.npy')):
        stem = fpath.stem[1:]  # strip leading 'p'
        scenario = next((k for k in LABEL_MAP if k in stem.lower()), None)
        device   = next((d for d in ['M7', 'X7', 'X300'] if stem.endswith(d)), None)
        if scenario is None or device is None:
            print(f'  WARNING: cannot parse {fpath.name}, skipping.')
            continue
        m = re.search(r'-(\d+)_', stem)
        session = int(m.group(1)) if m else 1
        arr = np.load(fpath)  # (W, T, K, C)
        records.append({
            'file':     fpath.name,
            'device':   device,
            'session':  session,
            'scenario': scenario,
            'label':    LABEL_MAP[scenario],
            'windows':  arr,
        })
    if not records:
        raise FileNotFoundError(
            f'No p*.npy files found under {processed_dir}. Run Notebook 1 first.')
    return records


ALL_RECORDS = load_all_records(PROCESSED_DIR)
ALL_DEVICES = ['M7', 'X7', 'X300']

print(f'Loaded {len(ALL_RECORDS)} trace records.')
print()
print(f"  {'Device':8s} {'Session':>8s} {'Label':>6s} {'Windows':>9s} {'File'}")
print(f"  {'-'*60}")
total = 0
for r in ALL_RECORDS:
    W = r['windows'].shape[0]
    total += W
    print(f"  {r['device']:8s} {r['session']:>8d} {r['label']:>6d} {W:>9d}  {r['file']}")
print(f"  {'-'*60}")
print(f"  Total windows: {total}")
print()
print('Window counts per device (expected to vary — no global cap applied):')
for dev in ALL_DEVICES:
    dev_recs = [r for r in ALL_RECORDS if r['device'] == dev]
    counts = [r['windows'].shape[0] for r in dev_recs]
    if counts:
        print(f'  {dev:8s} total={sum(counts)}  '
              f'min={min(counts)}  max={max(counts)}  traces={len(counts)})')

Loaded 27 trace records.

  Device    Session  Label   Windows File
  ------------------------------------------------------------
  M7              2      0        72  pvmatrix_empty-2_M7.npy
  M7              3      0        72  pvmatrix_empty-3_M7.npy
  M7              1      0        74  pvmatrix_empty_M7.npy
  M7              2      1        75  pvmatrix_moving-2_M7.npy
  M7              3      1        67  pvmatrix_moving-3_M7.npy
  M7              1      1        67  pvmatrix_moving_M7.npy
  M7              2      1        71  pvmatrix_stationary-2_M7.npy
  M7              3      1        73  pvmatrix_stationary-3_M7.npy
  M7              1      1        77  pvmatrix_stationary_M7.npy
  X300            2      0        68  pvmatrix_empty-2_X300.npy
  X300            3      0        54  pvmatrix_empty-3_X300.npy
  X300            1      0        67  pvmatrix_empty_X300.npy
  X300            2      1        73  pvmatrix_moving-2_X300.npy
  X300            3      1        48  pvmatr

---
## 2. CNN Encoder

Verbatim copy from Notebooks 2 and 3. **Do not modify** — all four notebooks must
use identical architecture for fair comparison.

In [ ]:
class CNNEncoder(nn.Module):
    """
    4-layer 2D CNN encoder
    Input : (N, C, T, K) → Output: (N, embed_dim)
    """
    def __init__(self, in_channels=N_CHANNELS, embed_dim=EMBED_DIM):
        super().__init__()
        def conv_block(ic, oc):
            return nn.Sequential(
                nn.Conv2d(ic, oc, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(oc),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2, ceil_mode=True),
            )
        self.layer1 = conv_block(in_channels, embed_dim)
        self.layer2 = conv_block(embed_dim,   embed_dim)
        self.layer3 = conv_block(embed_dim,   embed_dim)
        self.layer4 = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim, 3, padding=1, bias=False),
            nn.BatchNorm2d(embed_dim),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x.flatten(start_dim=1)  # (N, embed_dim)


# ── Instantiate ───────────────────────────────────────────────────────────────
encoder   = CNNEncoder(in_channels=N_CHANNELS, embed_dim=EMBED_DIM).to(TORCH_DEVICE)
optimizer = torch.optim.Adam(encoder.parameters(), lr=META_LR)

# ── Optional warm-start from Notebook 3 ──────────────────────────────────────
if WARMSTART_CKPT is not None:
    ckpt_path = CHECKPOINT_DIR / WARMSTART_CKPT
    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=TORCH_DEVICE)
        encoder.load_state_dict(ckpt['encoder_state_dict'])
        print(f'Warm-started encoder from {ckpt_path}')
        print(f'  (NB3 best fold: {ckpt["best_fold"]} F1={ckpt["best_f1"]:.3f})')
    else:
        print(f'WARNING: checkpoint {ckpt_path} not found — training from scratch.')
else:
    print('Training from random initialisation (WARMSTART_CKPT = None).')

n_params = sum(p.numel() for p in encoder.parameters())
print(f'Encoder parameters: {n_params:,}')

# Quick shape check
with torch.no_grad():
    _t = torch.randn(2, N_CHANNELS, WINDOW_SIZE, TARGET_SUBCARRIERS).to(TORCH_DEVICE)
    assert encoder(_t).shape == (2, EMBED_DIM)
print(f'PASS encoder output shape (2, {EMBED_DIM}) confirmed.')

Training from random initialisation (WARMSTART_CKPT = None).
Encoder parameters: 113,984
PASS encoder output shape (2, 64) confirmed.


---
## 3. Stratified Episode Sampler

An **episode** (a.k.a. *task*) is the atomic unit of ProtoNet training:

| Set | Size | Role |
|-----|------|------|
| Support | K × N_WAY windows | compute class prototypes |
| Query   | Q × N_WAY windows | compute loss / accuracy |

### Why stratified sampling?

Devices have different BFI frame rates (X7 > X300 > M7). If we pool all training
windows and draw uniformly, high-rate devices dominate each episode's support and
query sets. The encoder then learns embeddings tuned to whichever device happened
to contribute more windows, undermining the cross-device generalisation goal.

`sample_episode_stratified` accepts a `device_class_pool` — a two-level dict
`{device: {class: [window indices]}}` — and draws exactly `k` support and `q` query
windows **per device per class**. This enforces balanced device representation
within every episode without permanently discarding any windows.

A **small-data fallback** halves `k` and `q` when a device-class bucket has fewer
than `k + q` windows, returning `None` only if no valid episode can be formed.

In [ ]:
def windows_to_tensor(win_np: np.ndarray, device: torch.device) -> torch.Tensor:
    """
    Convert (M, T, K, C) channels-last numpy array → (M, C, T, K) float32 tensor.
    """
    return torch.from_numpy(win_np).float().permute(0, 3, 1, 2).to(device)

def build_device_class_pool(records, allowed_devices):
    """
    Build {device: {class_label: np.ndarray (W_dc, T, K, C)}} from a list of
    trace records, filtered to `allowed_devices`.

    Used by the stratified sampler and by kshot_evaluate_stratified.
    """
    pool = {}
    for dev in allowed_devices:
        dev_recs = [r for r in records if r['device'] == dev]
        pool[dev] = {}
        for cls in [0, 1]:
            cls_wins = [r['windows'] for r in dev_recs if r['label'] == cls]
            if cls_wins:
                pool[dev][cls] = np.concatenate(cls_wins, axis=0)
            else:
                pool[dev][cls] = np.empty((0,) + dev_recs[0]['windows'].shape[1:], dtype=np.float64)
    return pool

def sample_episode_stratified(device_class_pool, k, q, rng):
    """
    Sample one stratified ProtoNet episode.

    For each device in device_class_pool and each class, draw exactly k support
    and q query windows. All per-device, per-class contributions are equal in
    size, preventing high-rate devices from dominating the episode.

    Parameters
    ----------
    device_class_pool : {device: {class: np.ndarray (W_dc, T, K, C)}}
    k : support windows per device per class
    q : query windows per device per class
    rng : numpy random Generator

    Returns
    -------
    dict with keys: support_x, support_y, query_x, query_y
    OR None if any device-class bucket is too small even after fallback halving.
    """
    s_x, s_y, q_x, q_y = [], [], [], []
    for dev, cls_pool in device_class_pool.items():
        for cls, arr in cls_pool.items():
            _k, _q = k, q
            while len(arr) < _k + _q and (_k + _q) > 2:
                _k = max(1, _k // 2)
                _q = max(1, _q // 2)
            if len(arr) < _k + _q:
                return None
            chosen = rng.choice(len(arr), size=_k + _q, replace=False)
            rng.shuffle(chosen)
            s_x.append(arr[chosen[:_k]])
            s_y.append(np.full(_k, cls, dtype=np.int64))
            q_x.append(arr[chosen[_k:]])
            q_y.append(np.full(_q, cls, dtype=np.int64))
    return {'support_x': np.concatenate(s_x), 'support_y': np.concatenate(s_y),
            'query_x': np.concatenate(q_x), 'query_y': np.concatenate(q_y)}

print('build_device_class_pool and sample_episode_stratified defined.')
_pool = build_device_class_pool(ALL_RECORDS, ALL_DEVICES)
for dev, cls_d in _pool.items():
    for cls, arr in cls_d.items():
        print(f'  {dev} cls={cls}: {arr.shape[0]} windows')
_rng = np.random.default_rng(0)
_ep = sample_episode_stratified(_pool, k=1, q=1, rng=_rng)
if _ep:
    print(f'Test episode: support={_ep["support_x"].shape} query={_ep["query_x"].shape}')
else:
    print('Test episode: None')

build_device_class_pool and sample_episode_stratified defined.
  M7 cls=0: 218 windows
  M7 cls=1: 430 windows
  X7 cls=0: 283 windows
  X7 cls=1: 4248 windows
  X300 cls=0: 189 windows
  X300 cls=1: 361 windows
Test episode: support=(6, 5, 234, 5) query=(6, 5, 234, 5)


---
## 4. ProtoNet Loss and Episode Step

**Prototype computation:**
\[
\mathbf{c}_k = \frac{1}{|S_k|} \sum_{(x,y)\in S_k} f_\phi(x)
\]

**Classification distribution** (squared Euclidean, negated):
\[
p(y=k \mid x) = \frac{\exp\bigl(-d(f_\phi(x),\,\mathbf{c}_k)\bigr)}
    {\sum_{k'} \exp\bigl(-d(f_\phi(x),\,\mathbf{c}_{k'})\bigr)}
\]

**Loss** = cross-entropy over query labels, averaged across the episode.

Only the encoder \(f_\phi\) has learnable parameters; prototypes are
recomputed from scratch each episode.

In [ ]:
def proto_loss_and_acc(encoder, episode, device):
    """
    Run one ProtoNet episode: compute loss and query accuracy.

    Parameters
    ----------
    encoder : CNNEncoder in train or eval mode
    episode : dict from sample_episode_stratified
    device  : torch device

    Returns
    -------
    loss : scalar Tensor (differentiable)
    acc  : float query accuracy for this episode
    """
    sx = windows_to_tensor(episode['support_x'], device)
    sy = torch.from_numpy(episode['support_y']).long().to(device)
    qx = windows_to_tensor(episode['query_x'],   device)
    qy = torch.from_numpy(episode['query_y']).long().to(device)
    all_x   = torch.cat([sx, qx], dim=0)
    all_emb = encoder(all_x)
    n_s     = sx.shape[0]
    s_emb   = all_emb[:n_s]
    q_emb   = all_emb[n_s:]
    prototypes = torch.stack([s_emb[sy == cls].mean(dim=0) for cls in [0, 1]])
    dists = torch.cdist(q_emb.unsqueeze(0), prototypes.unsqueeze(0), p=2).squeeze(0) ** 2
    loss  = F.cross_entropy(-dists, qy)
    preds = dists.argmin(dim=1)
    acc   = float((preds == qy).float().mean().item())
    return loss, acc

print('proto_loss_and_acc defined.')
encoder.train()
_rng2 = np.random.default_rng(1)
_pool2 = build_device_class_pool(ALL_RECORDS, ALL_DEVICES)
_ep2 = sample_episode_stratified(_pool2, k=1, q=1, rng=_rng2)
if _ep2 is not None:
    _loss, _acc = proto_loss_and_acc(encoder, _ep2, TORCH_DEVICE)
    print(f'  Test episode: loss={_loss.item():.4f} acc={_acc:.2f}')
    assert _loss.requires_grad
    print('  PASS loss is differentiable.')

proto_loss_and_acc defined.
  Test episode: loss=0.4770 acc=1.00
  PASS loss is differentiable.


---
## 5. Meta-Training Loop

**Structure:** `META_EPOCHS` outer epochs, each containing `EPISODES_PER_EPOCH`
stratified episodes sampled from the **training split** (defined per experiment).
Early stopping on the running mean episode loss (patience = `META_PATIENCE`).

The returned `encoder` has its weights updated in-place.
The function is reused for both Experiment 2 and Experiment 3.

In [ ]:
def meta_train(encoder, optimizer, train_pool, k_shot, q_query, device,
               seed=RANDOM_SEED, label=''):
    """
    Run ProtoNet meta-training with stratified episode sampling.

    Parameters
    ----------
    encoder, optimizer : model and its Adam optimiser (modified in-place)
    train_pool : {device: {class: np.ndarray}} -- output of build_device_class_pool
    k_shot, q_query : per-device per-class episode sizes
    seed : RNG seed
    label : descriptive string for logging
    """
    rng = np.random.default_rng(seed)
    best_loss = float('inf')
    patience_ctr = 0
    history = []
    best_state = {k: v.cpu().clone() for k, v in encoder.state_dict().items()}

    hdr = f'  {"Epoch":>6} {"Mean Loss":>11} {"Mean Acc":>10} {"Pat":>6}'
    sep = '  ' + '-' * 40
    if label:
        print(f'Meta-training: {label}')
    print(hdr)
    print(sep)

    t0 = time.time()
    for epoch in range(1, META_EPOCHS + 1):
        epoch_losses, epoch_accs = [], []
        encoder.train()
        for _ in range(EPISODES_PER_EPOCH):
            ep = sample_episode_stratified(train_pool, k=k_shot, q=q_query, rng=rng)
            if ep is None:
                continue
            loss, acc = proto_loss_and_acc(encoder, ep, device)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
            epoch_accs.append(acc)

        if not epoch_losses:
            print(f'  {epoch:>6} [no valid episodes]')
            continue

        mean_loss = float(np.mean(epoch_losses))
        mean_acc  = float(np.mean(epoch_accs))
        history.append({'epoch': epoch, 'loss': mean_loss, 'acc': mean_acc})

        if mean_loss < best_loss:
            best_loss = mean_loss
            patience_ctr = 0
            best_state = {k: v.cpu().clone() for k, v in encoder.state_dict().items()}
        else:
            patience_ctr += 1

        if epoch == 1 or epoch % 10 == 0 or patience_ctr >= META_PATIENCE:
            elapsed = time.time() - t0
            print(f'  {epoch:>6} {mean_loss:>11.4f} {mean_acc:>10.3f} {patience_ctr:>3}/{META_PATIENCE} [{elapsed:.0f}s]')
        if patience_ctr >= META_PATIENCE:
            print(f'  Early stop at epoch {epoch}.')
            break

    encoder.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    print(f'  Best episode loss: {best_loss:.4f}')

    # Store in global log dict for later figure generation
    run_key = label if label else f'run_{int(time.time())}'
    META_TRAIN_LOGS[run_key] = {'best_loss': best_loss, 'history': history}

    return {'best_loss': best_loss, 'history': history}

print('meta_train defined.')

meta_train defined.


## 6. K-Shot Evaluation with Stratified Query Pool

After meta-training the encoder is frozen. For each K value:
1. For each device in the target split, sample K windows per class as the support set.
2. Compute class prototypes from support embeddings.
3. Classify all remaining target windows as queries.
4. Record accuracy, precision, recall, F1, and AUC-PR.

This is repeated over `N_EVAL_REPEATS` random support draws to reduce variance, then averaged. Per-device metrics are **macro-averaged** across devices to ensure each device contributes equally to the reported score regardless of window count differences.

AUC-PR (area under the precision-recall curve) is a threshold-independent metric preferred for
imbalanced binary classification. It is computed from query probabilities derived from
the Softmax over negative squared-Euclidean distances to class prototypes.

**Evaluation cap (optional robustness check):** By default `Q_EVAL_CAP = None` and fairness is achieved via macro-averaging. If set to an integer, each device's remaining query windows are subsampled to `eval_cap` before scoring — use this to verify that conclusions hold when pool sizes are exactly matched. The cap is symmetric, deterministic, and applied only at evaluation time; it does not affect the training pool.

In [ ]:
N_EVAL_REPEATS = 20  # number of random support draws per K


def compute_eval_cap(target_records, k):
    """
    Compute the per-device query cap for a given k and target split.

    Returns the minimum (total target windows for a device − k per class)
    across all devices present in target_records, or None if only one
    device is present (cap not needed).
    """
    devices_present = list({r['device'] for r in target_records})
    if len(devices_present) <= 1:
        return None
    per_dev_query = []
    for dev in devices_present:
        dev_recs = [r for r in target_records if r['device'] == dev]
        total = sum(r['windows'].shape[0] for r in dev_recs)
        per_dev_query.append(max(0, total - k * 2))
    return max(1, min(per_dev_query))

def kshot_evaluate_stratified(encoder, target_records, target_devices, k_list, device,
                               n_repeats=N_EVAL_REPEATS, seed=RANDOM_SEED, global_cap=Q_EVAL_CAP):
    """
    Evaluate frozen encoder on target split using stratified ProtoNet adaptation.

    Parameters
    ----------
    target_records  : list of trace record dicts from the target split
    target_devices  : devices present in the target split
    k_list          : list of K values to evaluate
    global_cap      : override for per-device query cap (None = compute per k)
    """
    encoder.eval()
    rng     = np.random.default_rng(seed)
    results = {}

    all_target_wins = np.concatenate([r['windows'] for r in target_records], axis=0)
    all_target_lbls = np.concatenate(
        [np.full(r['windows'].shape[0], r['label'], dtype=np.int64) for r in target_records])
    all_target_devs = np.concatenate(
        [[r['device']] * r['windows'].shape[0] for r in target_records])

    with torch.no_grad():
        t_x   = windows_to_tensor(all_target_wins, device)
        t_emb = encoder(t_x).cpu().numpy()

    for k in k_list:
        cap = global_cap if global_cap is not None else compute_eval_cap(target_records, k)
        rep_metrics = defaultdict(list)

        for rep in range(n_repeats):
            s_idx_all, q_idx_all = [], []
            valid = True
            for dev in target_devices:
                dev_mask = all_target_devs == dev
                dev_idxs = np.where(dev_mask)[0]
                s_dev, q_dev = [], []
                for cls in [0, 1]:
                    cls_idx = dev_idxs[all_target_lbls[dev_idxs] == cls]
                    if len(cls_idx) < k + 1:
                        valid = False
                        break
                    chosen = rng.choice(cls_idx, size=k, replace=False)
                    chosen_set = set(chosen.tolist())
                    s_dev.extend(chosen.tolist())
                    q_dev.extend([i for i in cls_idx if i not in chosen_set])
                if not valid:
                    break
                if cap is not None and len(q_dev) > cap:
                    q_dev = rng.choice(q_dev, size=cap, replace=False).tolist()
                s_idx_all.extend(s_dev)
                q_idx_all.extend(q_dev)

            if not valid or not q_idx_all:
                continue

            s_emb = t_emb[s_idx_all]
            s_lbl = all_target_lbls[s_idx_all]
            q_emb = t_emb[q_idx_all]
            q_lbl = all_target_lbls[q_idx_all]

            protos = np.stack([s_emb[s_lbl == cls].mean(axis=0) for cls in [0, 1]])
            diffs = q_emb[:, None, :] - protos[None, :, :]
            dists = (diffs ** 2).sum(axis=-1)
            preds = dists.argmin(axis=1)

            eps = 1e-8
            tp  = int(((preds == 1) & (q_lbl == 1)).sum())
            fp  = int(((preds == 1) & (q_lbl == 0)).sum())
            fn  = int(((preds == 0) & (q_lbl == 1)).sum())
            prec = tp / (tp + fp + eps)
            rec  = tp / (tp + fn + eps)
            f1   = 2 * prec * rec / (prec + rec + eps)
            acc  = float((preds == q_lbl).mean())
            # Compute P(occupied) from Softmax over negative distances
            # Use log-sum-exp trick for numerical stability: subtract max to prevent overflow/underflow
            neg_dists = -dists
            neg_dists_shifted = neg_dists - neg_dists.max(axis=1, keepdims=True)
            probs = np.exp(neg_dists_shifted)
            probs = probs / probs.sum(axis=1, keepdims=True)
            prob_pos = probs[:, 1]  # P(occupied)

            # Guard against NaN in probabilities
            if np.any(np.isnan(prob_pos)):
                prob_pos = np.nan_to_num(prob_pos, nan=0.5)

            rep_metrics['accuracy'].append(acc)
            rep_metrics['precision'].append(float(prec))
            rep_metrics['recall'].append(float(rec))
            rep_metrics['f1'].append(float(f1))
            if len(set(q_lbl)) > 1 and not np.all(np.isnan(prob_pos)):
                rep_metrics['auc_pr'].append(float(average_precision_score(q_lbl, prob_pos)))

        # Ensure auc_pr key exists even if only one class was present
        if 'auc_pr' not in rep_metrics:
            rep_metrics['auc_pr'] = []

        results[k] = {
            **{m: float(np.mean(v)) if v else float('nan') for m, v in rep_metrics.items()},
            **{f'{m}_std': float(np.std(v)) if v else float('nan') for m, v in rep_metrics.items()},
            'f1_raw': list(rep_metrics['f1']),
            'eval_cap': cap,
        }
    return results

print('kshot_evaluate_stratified defined.')

kshot_evaluate_stratified defined.


---
## 7. Experiment 2 — Cross-Device Few-Shot Generalisation

**Protocol (leave-one-device-out):**
- Train on 2 devices (all positions and scenarios).
- Adapt + evaluate on the 3rd (held-out) device using K-shot support.
- Repeat with each of the 3 devices as the target.

**Stratification in this experiment:** Training episodes draw equally from the two
training devices per class. The target device is fully held out from training.

Each run re-initialises the encoder to avoid leakage between target-device runs.

In [12]:
exp2_results = {}

for target_dev in ALL_DEVICES:
    train_devs  = [d for d in ALL_DEVICES if d != target_dev]
    train_recs  = [r for r in ALL_RECORDS if r['device'] in train_devs]
    target_recs = [r for r in ALL_RECORDS if r['device'] == target_dev]

    print()
    print('=' * 66)
    print(f'Experiment 2 | target={target_dev}  train={train_devs}')
    print('=' * 66)

    train_pool = build_device_class_pool(train_recs, train_devs)

    torch.manual_seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)
    enc_2 = CNNEncoder(N_CHANNELS, EMBED_DIM).to(TORCH_DEVICE)
    opt_2 = torch.optim.Adam(enc_2.parameters(), lr=META_LR)

    meta_train(enc_2, opt_2, train_pool, K_SHOT_TRAIN, Q_QUERY, TORCH_DEVICE,
               label=f'train on {train_devs} -> target {target_dev}')

    exp2_results[target_dev] = kshot_evaluate_stratified(
        enc_2, target_recs, [target_dev], K_SHOT_EVAL, TORCH_DEVICE)

    for k, m in exp2_results[target_dev].items():
        print(f'  K={k}: F1={m["f1"]:.3f}+-{m["f1_std"]:.3f}  AUC-PR={m["auc_pr"]:.3f}')

print()
print('Experiment 2 complete.')


Experiment 2 | target=M7  train=['X7', 'X300']
Meta-training: train on ['X7', 'X300'] -> target M7
   Epoch   Mean Loss   Mean Acc    Pat
  ----------------------------------------
       1      0.0620      0.979   0/100 [9s]
      10      0.0000      1.000   0/100 [84s]
      20      0.0000      1.000   2/100 [166s]
      30      0.0000      1.000   1/100 [248s]
      40      0.0000      1.000   0/100 [331s]
      50      0.0000      1.000   0/100 [413s]
      60      0.0000      1.000   2/100 [496s]
      70      0.0030      0.999   4/100 [579s]
      80      0.0000      1.000  14/100 [662s]
      90      0.0000      1.000  24/100 [746s]
     100      0.0000      1.000  34/100 [829s]
  Best episode loss: 0.0000
  K=1: F1=0.677+-0.167  AUC-PR=0.853
  K=3: F1=0.762+-0.098  AUC-PR=0.896
  K=5: F1=0.753+-0.163  AUC-PR=0.902
  K=10: F1=0.773+-0.161  AUC-PR=0.907

Experiment 2 | target=X7  train=['M7', 'X300']
Meta-training: train on ['M7', 'X300'] -> target X7
   Epoch   Mean Loss   Mean

---
## 8. Experiment 3 — Cross-Position Few-Shot Generalisation

**Protocol (leave-one-session-out, per device):**
- For each device separately, run 3 folds holding out one session at a time.
- Meta-train on 2 sessions; adapt + evaluate on the 3rd using K-shot support.
- This tests whether the encoder generalises across measurement positions
  within the same device.

**Stratification in this experiment:** Within a single-device fold, all training
windows come from one device so cross-device stratification is not needed.
However, the same `sample_episode_stratified` function is used for code consistency;
the pool will contain only one device key.

**Evaluation cap:** Applied per fold to the minimum remaining window count
per session in the target split, ensuring equal session contributions to metrics.

In [13]:
EXP3_FOLDS = [
    {'name': 'Fold 1', 'test_session': 1},
    {'name': 'Fold 2', 'test_session': 2},
    {'name': 'Fold 3', 'test_session': 3},
]

exp3_results = {}

for dev in ALL_DEVICES:
    exp3_results[dev] = {}
    for fold in EXP3_FOLDS:
        train_recs = [r for r in ALL_RECORDS
                      if r['device'] == dev and r['session'] != fold['test_session']]
        test_recs  = [r for r in ALL_RECORDS
                      if r['device'] == dev and r['session'] == fold['test_session']]

        if not train_recs or not test_recs:
            continue

        train_pool = build_device_class_pool(train_recs, [dev])

        torch.manual_seed(RANDOM_SEED)
        enc_3 = CNNEncoder(N_CHANNELS, EMBED_DIM).to(TORCH_DEVICE)
        opt_3 = torch.optim.Adam(enc_3.parameters(), lr=META_LR)

        meta_train(enc_3, opt_3, train_pool, K_SHOT_TRAIN, Q_QUERY, TORCH_DEVICE,
                   label=f'{dev} {fold["name"]}')

        exp3_results[dev][fold['name']] = kshot_evaluate_stratified(
            enc_3, test_recs, [dev], K_SHOT_EVAL, TORCH_DEVICE)

    for k in K_SHOT_EVAL:
        f1s = [exp3_results[dev][f][k]['f1'] for f in exp3_results[dev] if k in exp3_results[dev][f]]
        if f1s:
            print(f'  {dev} K={k}: F1={np.mean(f1s):.3f}')

print()
print('Experiment 3 complete.')

Meta-training: M7 Fold 1
   Epoch   Mean Loss   Mean Acc    Pat
  ----------------------------------------
       1      0.0256      0.993   0/100 [5s]
      10      0.0000      1.000   0/100 [47s]
      20      0.0000      1.000   2/100 [93s]
      30      0.0000      1.000   0/100 [139s]
      40      0.0000      1.000   2/100 [185s]
      50      0.0000      1.000   0/100 [232s]
      60      0.0000      1.000   0/100 [278s]
      70      0.0001      1.000   5/100 [324s]
      80      0.0000      1.000  15/100 [370s]
      90      0.0000      1.000  25/100 [417s]
     100      0.0000      1.000  35/100 [463s]
  Best episode loss: 0.0000
Meta-training: M7 Fold 2
   Epoch   Mean Loss   Mean Acc    Pat
  ----------------------------------------
       1      0.0303      0.992   0/100 [5s]
      10      0.0000      1.000   0/100 [46s]
      20      0.0000      1.000   0/100 [92s]
      30      0.0000      1.000   0/100 [139s]
      40      0.0000      1.000   1/100 [185s]
      50      

---
## 9. Results Summary and Tables

In [4]:
from scipy.stats import wilcoxon
import io, json
from pathlib import Path

RESULTS_DIR = Path('results')
K_LIST = [1, 3, 5, 10]
ALL_DEVICES = ['M7', 'X7', 'X300']
ALL_METRICS = ['accuracy', 'precision', 'recall', 'f1', 'auc_pr']

SEP  = '  ' + '-' * 74
RULER = '=' * 82
HDR  = '  {:>4s} {:>10s} {:>11s} {:>8s} {:>8s} {:>8s}'
ROW  = '  {:>4s} {:>10.3f} {:>11.3f} {:>8.3f} {:>8.3f} {:>8.3f}'
ROW_K = '  {:>4d} {:>10.3f} {:>11.3f} {:>8.3f} {:>8.3f} {:>8.3f}'
COL_LABELS = ('K', 'Accuracy', 'Precision', 'Recall', 'F1', 'AUC-PR')

# ── Load from saved JSON files ────────────────────────────────────────────
with open(RESULTS_DIR / 'exp2_cross_device.json') as f:
    raw_exp2 = json.load(f)   # keys are strings like "M7", with sub-keys "1", "3", etc.
with open(RESULTS_DIR / 'exp3_cross_position.json') as f:
    raw_exp3 = json.load(f)

# Convert string K-keys to int for both experiments
def int_keys(d):
    """Recursively convert dict keys that look like integers to int."""
    if not isinstance(d, dict):
        return d
    new = {}
    for k, v in d.items():
        try:
            new[int(k)] = int_keys(v)
        except (ValueError, TypeError):
            new[k] = int_keys(v)
    return new

exp2_results = int_keys(raw_exp2)  # now keys are ints
exp3_results = int_keys(raw_exp3)

print(f'Loaded exp2_results: devices={list(exp2_results.keys())}')
print(f'Loaded exp3_results: devices={list(exp3_results.keys())}')
for dev in ALL_DEVICES:
    n_folds = len(exp3_results.get(dev, {}))
    print(f'  {dev}: {n_folds} folds')

# ── Helper functions ──────────────────────────────────────────────────────
def collect_per_k(results_dict, klist):
    avg = {}
    for k in klist:
        vals = {m: [] for m in ALL_METRICS}
        for _, tgt_m in results_dict.items():
            if k not in tgt_m:
                continue
            for metric in ALL_METRICS:
                v = tgt_m[k].get(metric, float('nan'))
                if not np.isnan(v):
                    vals[metric].append(v)
        avg[k] = {m: float(np.mean(v)) if v else float('nan') for m, v in vals.items()}
    return avg

def collect_device_grand(results_dict, klist):
    grand = {}
    for dev, tgt_m in results_dict.items():
        vals = {m: [] for m in ALL_METRICS}
        for k in klist:
            if k not in tgt_m:
                continue
            for metric in ALL_METRICS:
                v = tgt_m[k].get(metric, float('nan'))
                if not np.isnan(v):
                    vals[metric].append(v)
        grand[dev] = {m: float(np.mean(v)) if v else float('nan') for m, v in vals.items()}
    return grand

def print_header(buf, title):
    buf.write(f'\n{RULER}\n  {title}\n{RULER}\n')

def print_metric_header(buf):
    buf.write(HDR.format(*COL_LABELS) + '\n')
    buf.write(SEP + '\n')

# ── Summary tables ────────────────────────────────────────────────────────
summary_buf = io.StringIO()
exp2_per_k   = collect_per_k(exp2_results, K_LIST)
exp2_dev_grand = collect_device_grand(exp2_results, K_LIST)

print_header(summary_buf, 'EXPERIMENT 2 - CROSS-DEVICE FEW-SHOT LEARNING')
print_header(summary_buf, 'Grand Mean (3 target devices)')
print_metric_header(summary_buf)
for k in K_LIST:
    m = exp2_per_k[k]
    summary_buf.write(ROW_K.format(k, m['accuracy'], m['precision'], m['recall'], m['f1'], m['auc_pr']) + '\n')
summary_buf.write(SEP + '\n\n  Per-device Grand Mean:\n')
print_metric_header(summary_buf)
for dev in ALL_DEVICES:
    if dev in exp2_dev_grand:
        m = exp2_dev_grand[dev]
        summary_buf.write(ROW.format(dev, m['accuracy'], m['precision'], m['recall'], m['f1'], m['auc_pr']) + '\n')
summary_buf.write(RULER + '\n')

flat3 = {f'{dev}_{fn}': m for dev in ALL_DEVICES for fn, m in exp3_results.get(dev, {}).items()}
exp3_per_k = collect_per_k(flat3, K_LIST)
exp3_dev_grand = collect_device_grand(exp3_results, K_LIST)

print_header(summary_buf, 'EXPERIMENT 3 - CROSS-POSITION FEW-SHOT LEARNING')
print_header(summary_buf, 'Grand Mean (3 devices x 3 folds)')
print_metric_header(summary_buf)
for k in K_LIST:
    m = exp3_per_k[k]
    summary_buf.write(ROW_K.format(k, m['accuracy'], m['precision'], m['recall'], m['f1'], m['auc_pr']) + '\n')
summary_buf.write(SEP + '\n\n  Per-device Grand Mean:\n')
print_metric_header(summary_buf)
for dev in ALL_DEVICES:
    if dev in exp3_dev_grand:
        m = exp3_dev_grand[dev]
        summary_buf.write(ROW.format(dev, m['accuracy'], m['precision'], m['recall'], m['f1'], m['auc_pr']) + '\n')
summary_buf.write(RULER + '\n')

# ── Detailed tables ───────────────────────────────────────────────────────
detail_buf = io.StringIO()
print_header(detail_buf, 'EXPERIMENT 2 - CROSS-DEVICE FSL (Detailed)')
for dev in ALL_DEVICES:
    if dev not in exp2_results:
        continue
    dev_metrics = exp2_results[dev]
    dev_vals = {m: [] for m in ALL_METRICS}
    detail_buf.write(f'\n  Target device: {dev}\n')
    print_metric_header(detail_buf)
    for k in K_LIST:
        if k not in dev_metrics:
            continue
        m = dev_metrics[k]
        auc = m.get('auc_pr', float('nan'))
        detail_buf.write(ROW_K.format(k, m['accuracy'], m['precision'], m['recall'], m['f1'], auc) + '\n')
        for metric in ALL_METRICS:
            v = m.get(metric, float('nan'))
            if not np.isnan(v):
                dev_vals[metric].append(v)
    mean_row = {m: float(np.mean(dev_vals[m])) if dev_vals[m] else float('nan') for m in ALL_METRICS}
    detail_buf.write(ROW.format('Mean', mean_row['accuracy'], mean_row['precision'], mean_row['recall'], mean_row['f1'], mean_row['auc_pr']) + '\n')
    detail_buf.write(SEP + '\n')
detail_buf.write(RULER + '\n')

print_header(detail_buf, 'EXPERIMENT 3 - CROSS-POSITION FSL (Detailed)')
for dev in ALL_DEVICES:
    if dev not in exp3_results or not exp3_results[dev]:
        continue
    detail_buf.write(f'\n  Device: {dev}\n  {"-" * 50}\n')
    fold_names = sorted(exp3_results[dev].keys())
    for fold_name in fold_names:
        fold_metrics = exp3_results[dev][fold_name]
        fold_vals = {m: [] for m in ALL_METRICS}
        detail_buf.write(f'  {fold_name}\n')
        print_metric_header(detail_buf)
        for k in K_LIST:
            if k not in fold_metrics:
                continue
            m = fold_metrics[k]
            auc = m.get('auc_pr', float('nan'))
            detail_buf.write(ROW_K.format(k, m['accuracy'], m['precision'], m['recall'], m['f1'], auc) + '\n')
            for metric in ALL_METRICS:
                v = m.get(metric, float('nan'))
                if not np.isnan(v):
                    fold_vals[metric].append(v)
        fm = {m: float(np.mean(fold_vals[m])) if fold_vals[m] else float('nan') for m in ALL_METRICS}
        detail_buf.write(ROW.format('Mean', fm['accuracy'], fm['precision'], fm['recall'], fm['f1'], fm['auc_pr']) + '\n')
        detail_buf.write(SEP + '\n')

exp3_grand = collect_per_k(flat3, K_LIST)
print_header(detail_buf, 'Exp 3 - Grand Mean (All Devices x All Folds)')
print_metric_header(detail_buf)
for k in K_LIST:
    m = exp3_grand[k]
    detail_buf.write(ROW_K.format(k, m['accuracy'], m['precision'], m['recall'], m['f1'], m['auc_pr']) + '\n')
detail_buf.write(RULER + '\n')

# ═══════════════════════════════════════════════════════════════════════════════
#  CORRECTED STATISTICAL TESTS
# ═══════════════════════════════════════════════════════════════════════════════

def wilcoxon_fold_level(results_dict, klist, label):
    """
    Wilcoxon signed-rank on FOLD-LEVEL MEAN F1.
    Each fold contributes ONE paired observation (mean F1 at K=a vs mean F1 at K=b).
    """
    buf = io.StringIO()
    pairs = [(klist[i], klist[i+1]) for i in range(len(klist) - 1)]
    bonf_alpha = 0.05 / len(pairs)
    
    buf.write(f'\nWilcoxon signed-rank test (fold-level means, Bonferroni-corrected) - {label}\n')
    buf.write(f'  family size = {len(pairs)}  raw α = 0.05  corrected α = {bonf_alpha:.5f}\n')
    buf.write('-' * 66 + '\n')
    buf.write(f"  {'Comparison':>12s} {'W stat':>10s} {'p-value':>10s} {'Significant':>12s}\n")
    buf.write('  ' + '-' * 54 + '\n')
    
    valid_pairs = 0
    for ka, kb in pairs:
        fold_means_a, fold_means_b = [], []
        for fold_key in results_dict:
            fold_m = results_dict[fold_key]
            if ka in fold_m and kb in fold_m:
                raw_a = fold_m[ka].get('f1_raw', [])
                raw_b = fold_m[kb].get('f1_raw', [])
                if len(raw_a) > 0 and len(raw_b) > 0:
                    fold_means_a.append(float(np.mean(raw_a)))
                    fold_means_b.append(float(np.mean(raw_b)))
        
        fold_means_a, fold_means_b = np.array(fold_means_a), np.array(fold_means_b)
        n_folds = len(fold_means_a)
        
        if n_folds < 6:
            buf.write(f"  {f'K={ka} vs K={kb}':>12s} {'—':>10s} {'—':>10s} {'n={n_folds}<6':>12s}\n")
            continue
        
        diffs = fold_means_b - fold_means_a
        if np.all(np.abs(diffs) < 1e-12):
            buf.write(f"  {f'K={ka} vs K={kb}':>12s} {'—':>10s} {'—':>10s} {'all ties':>12s}\n")
            continue
        
        stat, p = wilcoxon(fold_means_a, fold_means_b, alternative='two-sided')
        sig = 'YES *' if p < bonf_alpha else 'no'
        valid_pairs += 1
        buf.write(f"  {f'K={ka} vs K={kb}':>12s} {stat:>10.1f} {p:>10.4f} {sig:>12s}  (n={n_folds} folds)\n")
    
    buf.write('-' * 66 + '\n')
    buf.write(f'  Valid comparisons: {valid_pairs}/{len(pairs)}\n')
    return buf.getvalue()


# ── Exp 3: 9 fold-level pairs → Wilcoxon signed-rank ─────────────────────
# 9 independent folds: 3 devices × 3 test sessions = 9 paired observations
exp3_stat_text = wilcoxon_fold_level(flat3, K_LIST, 'Exp 3 Cross-Position FSL')

# ── Exp 2: 3 runs only → Descriptive, no significance test ───────────────
# Only 3 independent observations (one per target device).
# No nonparametric test is valid with n=3.
exp2_stat_buf = io.StringIO()
exp2_stat_buf.write('\n' + '=' * 66 + '\n')
exp2_stat_buf.write('  Exp 2 Statistical Testing — DESCRIPTIVE ONLY\n')
exp2_stat_buf.write('=' * 66 + '\n')
exp2_stat_buf.write('  Cross-device FSL has only n=3 independent runs\n')
exp2_stat_buf.write('  (one per target device). No significance test is\n')
exp2_stat_buf.write('  valid at this sample size. Results reported\n')
exp2_stat_buf.write('  descriptively. Significance testing deferred to\n')
exp2_stat_buf.write('  future work with more held-out devices.\n')
exp2_stat_buf.write('=' * 66 + '\n')

exp2_stat_text = exp2_stat_buf.getvalue()

# ═══════════════════════════════════════════════════════════════════════════════
#  PRINT TO CONSOLE
# ═══════════════════════════════════════════════════════════════════════════════

print(summary_buf.getvalue())
print(exp2_stat_text)
print(exp3_stat_text)
print()
print(detail_buf.getvalue())


# ═══════════════════════════════════════════════════════════════════════════════
#  SAVE TO FILES
# ═══════════════════════════════════════════════════════════════════════════════

with open(RESULTS_DIR / 'exp2-3_summary.txt', 'w') as f:
    f.write(summary_buf.getvalue())
    f.write(exp2_stat_text)
    f.write(exp3_stat_text)
    f.write('\n')
    f.write('==== End of Summary Tables (Chapter 4 ready) ====\n')

with open(RESULTS_DIR / 'exp2-3_detailed.txt', 'w') as f:
    f.write(detail_buf.getvalue())
    f.write('\n')
    f.write('==== End of Detailed Tables (Appendix ready) ====\n')

print('Saved: exp2-3_summary.txt and exp2-3_detailed.txt')

Loaded exp2_results: devices=['M7', 'X7', 'X300']
Loaded exp3_results: devices=['M7', 'X7', 'X300']
  M7: 3 folds
  X7: 3 folds
  X300: 3 folds

  EXPERIMENT 2 - CROSS-DEVICE FEW-SHOT LEARNING

  Grand Mean (3 target devices)
     K   Accuracy   Precision   Recall       F1   AUC-PR
  --------------------------------------------------------------------------
     1      0.624       0.862    0.565    0.648    0.863
     3      0.687       0.862    0.647    0.720    0.881
     5      0.696       0.864    0.656    0.726    0.885
    10      0.728       0.870    0.711    0.772    0.896
  --------------------------------------------------------------------------

  Per-device Grand Mean:
     K   Accuracy   Precision   Recall       F1   AUC-PR
  --------------------------------------------------------------------------
    M7      0.705       0.846    0.688    0.742    0.890
    X7      0.823       0.984    0.816    0.879    0.991
  X300      0.522       0.763    0.430    0.529    0.764

  E

---
## 10. Save Results and Final Checkpoint

In [15]:
def serialise(d):
    if isinstance(d, dict):
        return {str(k): serialise(v) for k, v in d.items()}
    return d

with open(RESULTS_DIR / 'exp2_cross_device.json', 'w') as f:
    json.dump(serialise(exp2_results), f, indent=2)
with open(RESULTS_DIR / 'exp3_cross_position.json', 'w') as f:
    json.dump(serialise(exp3_results), f, indent=2)

# Save final encoder checkpoint
ckpt_path = CHECKPOINT_DIR / 'nb4_protonet_final_encoder.pt'
torch.save({
    'encoder_state_dict': {k: v.cpu() for k, v in enc_3.state_dict().items()},
    'config': {'n_channels': N_CHANNELS, 'embed_dim': EMBED_DIM,
               'window_size': WINDOW_SIZE, 'target_subcarriers': TARGET_SUBCARRIERS},
    'hyperparams': {'k_shot_train': K_SHOT_TRAIN, 'q_query_per_device': Q_QUERY,
                    'meta_epochs': META_EPOCHS, 'episodes_per_epoch': EPISODES_PER_EPOCH,
                    'meta_lr': META_LR, 'random_seed': RANDOM_SEED, 'stratified_sampler': True},
}, ckpt_path)
print(f'Saved: {ckpt_path}')

# Save meta-training loss logs for figure generation
with open(RESULTS_DIR / 'meta_train_logs.json', 'w') as f:
    json.dump(META_TRAIN_LOGS, f, indent=2)
print(f'Saved: {RESULTS_DIR / "meta_train_logs.json"}')
print()
print('--- Notebook 4 results saved. Proceed to Cell 11 for figures. ---')

Saved: checkpoints\nb4_protonet_final_encoder.pt
Saved: results\meta_train_logs.json

--- Notebook 4 results saved. Proceed to Cell 11 for figures. ---


---
## 11. Generate Publication-Ready Figures for Chapter 4

Creates the following figures from saved results:
- **Figure 4.2**: Meta-training loss curves (epoch vs mean loss per run)
- **Figure 4.3**: Exp2 K-shot sensitivity (F1 vs K, one line per target device)
- **Figure 4.4**: Exp3 F1 at K=10 grouped bar chart (per device x per fold)
- **Figure 4.5**: Exp3 K-shot sensitivity (F1 vs K, one line per device)

Figures are saved to `results/figures/` and can be inserted directly into the
Chapter 4 document. This cell reads only from saved JSON files, so it can be
run independently after the rest of the notebook completes.

In [2]:
import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
matplotlib.rcParams['pdf.fonttype'] = 42  # TrueType, so Inkscape can render text in PDF→EMF
matplotlib.rcParams['ps.fonttype'] = 42
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
import re
import subprocess

RESULTS_DIR = Path('results')
FIG_DIR     = RESULTS_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Adjust this path if your Inkscape installation is elsewhere
INKSCAPE_PATH = r'C:\Program Files\Inkscape\inkscape.com'

DEVICES      = ['M7', 'X7', 'X300']
DEVICE_COLORS = {'M7': '#2196F3', 'X7': '#FF9800', 'X300': '#4CAF50'}
DEVICE_DISPLAY_NAMES = {'M7': 'M7', 'X7': 'X7', 'X300': 'X300 Pro'}
K_LIST = [1, 3, 5, 10]

# Load saved data
with open(RESULTS_DIR / 'exp2_cross_device.json') as f:
    exp2 = json.load(f)
with open(RESULTS_DIR / 'exp3_cross_position.json') as f:
    exp3 = json.load(f)
with open(RESULTS_DIR / 'meta_train_logs.json') as f:
    logs = json.load(f)

print(f'Loaded exp2 ({len(exp2)} devices), exp3 ({len(exp3)} devices), and logs ({len(logs)} runs)')


# ===== HELPER: Save figure as PDF + PNG + EMF (EMF via PDF→Inkscape) =====
def save_figure(fig, base_path, dpi=600, emf=True):
    """Save a matplotlib figure as PDF + PNG + EMF (EMF via PDF→Inkscape)."""
    base = Path(base_path).with_suffix('')
    pdf_path = base.with_suffix('.pdf')
    png_path = base.with_suffix('.png')
    emf_path = base.with_suffix('.emf')
    
    # PDF (vector, kept as a deliverable)
    fig.savefig(pdf_path)
    
    # PNG (raster, for previewing)
    fig.savefig(png_path, dpi=dpi)
    if not emf:
        print(f'Saved: {base.name}.{{pdf,png}}')
        return
    
    
    # PDF → EMF via Inkscape (scoop shim reached via the PATH fallback)
    inkscape = INKSCAPE_PATH
    if not Path(inkscape).exists():
        inkscape = r'C:\Program Files\Inkscape\bin\inkscape.com'
    if not Path(inkscape).exists():
        inkscape = 'inkscape'  # fallback to PATH
    
    result = subprocess.run(
        [inkscape, str(pdf_path), '--export-type=emf', f'--export-filename={emf_path}'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'Warning: Inkscape PDF→EMF failed for {pdf_path.name}')
        print(result.stderr)
    
    print(f'Saved: {base.name}.{{pdf,png,emf}}')


# ===== HELPER =====
def get_device_metric(exp_dict, dev, k, metric):
    dev_data = exp_dict.get(dev, {})
    entry = dev_data.get(str(k), dev_data.get(k, None))
    if entry is None:
        return float('nan')
    return entry.get(metric, float('nan'))

# ===== FIGURE 4.1: Normalization verification (boxplots) =====
print('Generating Figure 4.1: Normalization visualization...')
PROCESSED_DIR = Path('data/processed')
RAW_DATA_DIR  = Path('data')  # Raw complex-valued V-matrices

fig, axes = plt.subplots(3, 2, figsize=(10, 8))
# Order matches Notebook 1 §3 and paper Table 3.2
channel_labels = ['Re(ant0)', 'Re(ant1)', 'Re(ant2)', 'Im(ant0)', 'Im(ant1)']
channel_colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#F44336']

for row_idx, device in enumerate(DEVICES):
    # ── Load RAW data for 'Before Normalization' panel ────────────────
    raw_device_dir = RAW_DATA_DIR / device
    raw_files = sorted(raw_device_dir.glob('*.npy')) if raw_device_dir.is_dir() else []
    
    # ── Load first normalized file for 'After Normalization' panel ─────
    proc_device_dir = PROCESSED_DIR / device
    proc_files = sorted(proc_device_dir.glob('*.npy')) if proc_device_dir.is_dir() else []
    
    if not raw_files or not proc_files:
        print(f'  WARNING: Missing files for {device}, skipping.')
        continue

    # Load and process RAW data (before normalization):
    # Complex V-matrix → extract real/imag channels → window → flatten to frames
    raw_arr = np.load(raw_files[0])  # (P, K, M, Nss) complex
    # Truncate to TARGET_SUBCARRIERS (234) and keep first spatial stream
    raw_trunc = raw_arr[:, :TARGET_SUBCARRIERS, :, :TARGET_NSS]
    # Squeeze Nss axis: (P, 234, 3) → then split real/imag: (P, 234, 6)
    raw_squeeze = raw_trunc.squeeze(axis=-1)  # (P, 234, 3)
    raw_features = np.concatenate([raw_squeeze.real, raw_squeeze.imag], axis=-1)  # (P, 234, 6)
    # Drop Im(ant2) if only 5 channels used
    raw_features = raw_features[:, :, :N_CHANNELS]
    # Quality filter: remove NaN/Inf/zero-only frames
    nan_or_inf = np.all(np.isfinite(raw_features), axis=(1, 2))
    all_zero   = np.all(raw_features == 0, axis=(1, 2))
    valid_mask = nan_or_inf & ~all_zero
    raw_clean  = raw_features[valid_mask]  # (P_clean, 234, C)
    # Window: reshape to (W, T, K, C) and take first W*T frames
    W = raw_clean.shape[0] // WINDOW_SIZE
    usable = W * WINDOW_SIZE
    raw_windows = raw_clean[:usable].reshape(W, WINDOW_SIZE, TARGET_SUBCARRIERS, N_CHANNELS)
    raw_frames = raw_windows.reshape(-1, raw_windows.shape[2], raw_windows.shape[3])  # (W*T, K, C)

    # Load normalized data (after normalization):
    norm_windows = np.load(proc_files[0])  # (W, T, K, C)
    norm_frames = norm_windows.reshape(-1, norm_windows.shape[2], norm_windows.shape[3])  # (W*T, K, C)

    # ── Before Normalization ──────────────────────────────────────────
    ax = axes[row_idx][0]
    ax.set_ylabel(DEVICE_DISPLAY_NAMES[device], fontsize=10)

    data_before = [raw_frames[:, :, ch].flatten() for ch in range(raw_frames.shape[2])]
    bp_before = ax.boxplot(data_before, positions=range(1, raw_frames.shape[2] + 1),
                           patch_artist=True, widths=0.5,
                           medianprops=dict(color='black', linewidth=1.0))
    for patch, color in zip(bp_before['boxes'], channel_colors[:raw_frames.shape[2]]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)

    if row_idx == 0:
        ax.set_title('Before Normalization', fontsize=11)
    ax.set_xticks(range(1, raw_frames.shape[2] + 1))
    ax.set_xticklabels(channel_labels[:raw_frames.shape[2]], fontsize=7)
    if row_idx == 2:
        ax.set_xlabel('Channel', fontsize=9)

    # ── After Normalization ───────────────────────────────────────────
    ax2 = axes[row_idx][1]

    data_after = [norm_frames[:, :, ch].flatten() for ch in range(norm_frames.shape[2])]
    bp_after = ax2.boxplot(data_after, positions=range(1, norm_frames.shape[2] + 1),
                           patch_artist=True, widths=0.5,
                           medianprops=dict(color='black', linewidth=1.0))
    for patch, color in zip(bp_after['boxes'], channel_colors[:norm_frames.shape[2]]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)

    if row_idx == 0:
        ax2.set_title('After Per-Channel Z-Score Normalization', fontsize=11)
    ax2.set_xticks(range(1, norm_frames.shape[2] + 1))
    ax2.set_xticklabels(channel_labels[:norm_frames.shape[2]], fontsize=7)
    ax2.set_xlim(0, norm_frames.shape[2] + 1)

    # Reference lines at 0 and ±1 (meaningful only on normalized side)
    ax2.axhline(0, color='black', linestyle='--', linewidth=0.8)
    ax2.axhline(1, color='gray', linestyle=':', linewidth=0.8)
    ax2.axhline(-1, color='gray', linestyle=':', linewidth=0.8)

    if row_idx == 2:
        ax2.set_xlabel('Channel', fontsize=9)

# No title
fig.tight_layout()
save_figure(fig, FIG_DIR / 'fig4_1_normalization')
plt.close()

# ===== FIGURE 4.2 — Loss Curves =====
fig, ax = plt.subplots(figsize=(8, 4.5))
for run_key, run_data in logs.items():
    history = run_data.get('history', [])
    if not history:
        continue
    epochs = [h['epoch'] for h in history]
    losses = [h['loss'] for h in history]
    # Apply display-name mapping to labels
    label = run_key.split(' -> ')[0] if ' -> ' in run_key else run_key
    for k, v in DEVICE_DISPLAY_NAMES.items():
        label = label.replace(k, v)
    ax.plot(epochs, losses, lw=0.8, alpha=0.7, label=label)
ax.set_xlabel('Epoch')
ax.set_ylabel('Mean Episode Loss')
ax.legend(fontsize=6, loc='upper right', ncol=2)
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_figure(fig, FIG_DIR / 'fig4_2_loss_curves')
plt.close()

# ===== FIGURE 4.3a — Exp2 PR-AUC K-Shot =====
fig, ax = plt.subplots(figsize=(7, 4.5))
for dev in DEVICES:
    if dev not in exp2:
        continue
    vals, stds = [], []
    for k in K_LIST:
        vals.append(get_device_metric(exp2, dev, k, 'auc_pr'))
        entry = exp2[dev].get(str(k), exp2[dev].get(k, {}))
        std_val = entry.get('auc_pr_std', float('nan')) if isinstance(entry, dict) else float('nan')
        stds.append(std_val)
    ax.errorbar(K_LIST, vals, yerr=stds,
                color=DEVICE_COLORS[dev], marker='o', capsize=4,
                label=DEVICE_DISPLAY_NAMES[dev])
ax.set_xlabel('K (Support Samples Per Class)')
ax.set_ylabel('PR-AUC')
ax.set_xticks(K_LIST)
ax.set_ylim(0.6, 1.0)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_figure(fig, FIG_DIR / 'fig4_3a_exp2_pr_auc_kshot')
plt.close()

# ===== FIGURE 4.3b — Exp2 F1 K-Shot =====
fig, ax = plt.subplots(figsize=(7, 4.5))
for dev in DEVICES:
    if dev not in exp2:
        continue
    vals, stds = [], []
    for k in K_LIST:
        vals.append(get_device_metric(exp2, dev, k, 'f1'))
        entry = exp2[dev].get(str(k), exp2[dev].get(k, {}))
        std_val = entry.get('f1_std', float('nan')) if isinstance(entry, dict) else float('nan')
        stds.append(std_val)
    ax.errorbar(K_LIST, vals, yerr=stds,
                color=DEVICE_COLORS[dev], marker='o', capsize=4,
                label=DEVICE_DISPLAY_NAMES[dev])
ax.set_xlabel('K (Support Samples Per Class)')
ax.set_ylabel('F1')
ax.set_xticks(K_LIST)
ax.set_ylim(0.3, 1.0)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_figure(fig, FIG_DIR / 'fig4_3b_exp2_f1_kshot')
plt.close()

# ===== FIGURE 4.4a — Exp3 PR-AUC at K=10 Grouped Bar =====
x = np.arange(len(DEVICES))
width = 0.25
fold_styles = [
    {'label': 'Fold 1', 'hatch': '', 'alpha': 0.9, 'color': '#2196F3'},
    {'label': 'Fold 2', 'hatch': '///', 'alpha': 0.7, 'color': '#FF9800'},
    {'label': 'Fold 3', 'hatch': '...', 'alpha': 0.8, 'color': '#4CAF50'},
]

fig, ax = plt.subplots(figsize=(7, 4.5))
multiplier = 0
for fold_info in fold_styles:
    fold_name = fold_info['label']
    k10_vals = []
    for dev in DEVICES:
        if dev in exp3 and fold_name in exp3[dev]:
            entry = exp3[dev][fold_name].get('10', exp3[dev][fold_name].get(10, {}))
            k10_vals.append(entry.get('auc_pr', float('nan')))
        else:
            k10_vals.append(float('nan'))
    offset = width * multiplier
    ax.bar(x + offset, k10_vals, width,
           label=fold_info['label'], alpha=fold_info['alpha'],
           color=fold_info['color'], hatch=fold_info['hatch'])
    multiplier += 1
ax.set_xlabel('Target Device')
ax.set_ylabel('PR-AUC at K = 10')
ax.set_xticks(x + width)
ax.set_xticklabels([DEVICE_DISPLAY_NAMES[d] for d in DEVICES])
ax.set_ylim(0.6, 1.0)
ax.legend(loc='lower right')
ax.axhline(y=0.949, color='gray', linestyle='--', linewidth=0.8, label='Supervised Baseline (0.949)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
save_figure(fig, FIG_DIR / 'fig4_4a_exp3_pr_auc_k10_barchart', emf=False)
plt.close()

# ===== FIGURE 4.4b — Exp3 F1 at K=10 Grouped Bar =====
fig, ax = plt.subplots(figsize=(7, 4.5))
multiplier = 0
for fold_info in fold_styles:
    fold_name = fold_info['label']
    k10_vals = []
    for dev in DEVICES:
        if dev in exp3 and fold_name in exp3[dev]:
            entry = exp3[dev][fold_name].get('10', exp3[dev][fold_name].get(10, {}))
            k10_vals.append(entry.get('f1', float('nan')))
        else:
            k10_vals.append(float('nan'))
    offset = width * multiplier
    ax.bar(x + offset, k10_vals, width,
           label=fold_info['label'], alpha=fold_info['alpha'],
           color=fold_info['color'], hatch=fold_info['hatch'])
    multiplier += 1
ax.set_xlabel('Target Device')
ax.set_ylabel('F1 at K = 10')
ax.set_xticks(x + width)
ax.set_xticklabels([DEVICE_DISPLAY_NAMES[d] for d in DEVICES])
ax.set_ylim(0.4, 1.0)
ax.legend(loc='lower right')
ax.axhline(y=0.839, color='gray', linestyle='--', linewidth=0.8, label='Supervised Baseline (0.839)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
save_figure(fig, FIG_DIR / 'fig4_4b_exp3_f1_k10_barchart', emf=False)
plt.close()

# ===== FIGURE 4.5a — Exp3 PR-AUC K-Shot =====
fig, ax = plt.subplots(figsize=(7, 4.5))
for dev in DEVICES:
    if dev not in exp3:
        continue
    folds_dict = exp3[dev]
    vals, stds = [], []
    for k in K_LIST:
        k_aucs = []
        for fold_name in folds_dict:
            entry = folds_dict[fold_name].get(str(k), folds_dict[fold_name].get(k, {}))
            v = entry.get('auc_pr', float('nan'))
            if not np.isnan(v):
                k_aucs.append(v)
        if k_aucs:
            vals.append(float(np.mean(k_aucs)))
            stds.append(float(np.std(k_aucs, ddof=1)))
        else:
            vals.append(float('nan'))
            stds.append(float('nan'))
    ax.errorbar(K_LIST, vals, yerr=stds,
                color=DEVICE_COLORS[dev], marker='s', capsize=4,
                label=DEVICE_DISPLAY_NAMES[dev])
ax.set_xlabel('K (Support Samples Per Class)')
ax.set_ylabel('PR-AUC')
ax.set_xticks(K_LIST)
ax.set_ylim(0.8, 1.0)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_figure(fig, FIG_DIR / 'fig4_5a_exp3_pr_auc_kshot')
plt.close()

# ===== FIGURE 4.5b — Exp3 F1 K-Shot =====
fig, ax = plt.subplots(figsize=(7, 4.5))
for dev in DEVICES:
    if dev not in exp3:
        continue
    folds_dict = exp3[dev]
    vals, stds = [], []
    for k in K_LIST:
        k_f1s = []
        for fold_name in folds_dict:
            entry = folds_dict[fold_name].get(str(k), folds_dict[fold_name].get(k, {}))
            v = entry.get('f1', float('nan'))
            if not np.isnan(v):
                k_f1s.append(v)
        if k_f1s:
            vals.append(float(np.mean(k_f1s)))
            stds.append(float(np.std(k_f1s, ddof=1)))
        else:
            vals.append(float('nan'))
            stds.append(float('nan'))
    ax.errorbar(K_LIST, vals, yerr=stds,
                color=DEVICE_COLORS[dev], marker='s', capsize=4,
                label=DEVICE_DISPLAY_NAMES[dev])
ax.set_xlabel('K (Support Samples Per Class)')
ax.set_ylabel('F1')
ax.set_xticks(K_LIST)
ax.set_ylim(0.4, 1.0)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_figure(fig, FIG_DIR / 'fig4_5b_exp3_f1_kshot')
plt.close()

print()
print('All figures saved to results/figures/.')
print()
for ext in ('pdf', 'emf', 'png'):
    print(f'Figure list ({ext.upper()}):')
    for f in sorted(FIG_DIR.glob(f'fig4_*.{ext}')):
        print(f'  {f.name}')
    print()
print('--- Figure generation complete. ---')


Loaded exp2 (3 devices), exp3 (3 devices), and logs (12 runs)
Generating Figure 4.1: Normalization visualization...
Saved: fig4_1_normalization.{pdf,png,emf}
Saved: fig4_2_loss_curves.{pdf,png,emf}
Saved: fig4_3a_exp2_pr_auc_kshot.{pdf,png,emf}
Saved: fig4_3b_exp2_f1_kshot.{pdf,png,emf}
Saved: fig4_4a_exp3_pr_auc_k10_barchart.{pdf,png}
Saved: fig4_4b_exp3_f1_k10_barchart.{pdf,png}
Saved: fig4_5a_exp3_pr_auc_kshot.{pdf,png,emf}
Saved: fig4_5b_exp3_f1_kshot.{pdf,png,emf}

All figures saved to results/figures/.

Figure list (PDF):
  fig4_1_normalization.pdf
  fig4_2_loss_curves.pdf
  fig4_3a_exp2_pr_auc_kshot.pdf
  fig4_3b_exp2_f1_kshot.pdf
  fig4_4a_exp3_pr_auc_k10_barchart.pdf
  fig4_4b_exp3_f1_k10_barchart.pdf
  fig4_5a_exp3_pr_auc_kshot.pdf
  fig4_5b_exp3_f1_kshot.pdf

Figure list (EMF):
  fig4_1_normalization.emf
  fig4_2_loss_curves.emf
  fig4_3a_exp2_pr_auc_kshot.emf
  fig4_3b_exp2_f1_kshot.emf
  fig4_5a_exp3_pr_auc_kshot.emf
  fig4_5b_exp3_f1_kshot.emf

Figure list (PNG):
  fig4_1